# Continual RL on ManiSkill Push-T  —  Google Colab Runner

This notebook executes the full 4-stage Continual RL pipeline on Google Colab with Google Drive persistence.

| Section | Stage | What it does | Re-run after restart? |
|---------|-------|-------------|----------------------|
| 0 | Setup | GPU check, Drive mount, clone repo, install deps, smoke test | Yes (every restart) |
| 1 | T-I: Imitation Learning | Download demos, train Diffusion Policy | Only if training incomplete |
| 2 | T-II: Failure Analysis | Evaluate policy, classify failure modes, visualize | After T-I completes |
| 3 | T-III: LLM Reward Gen | Generate reward function from failure video via LLM | After T-II analysis |
| 4 | T-IV: PPO Fine-tuning | RL fine-tuning with LLM reward, evaluate improvement | After T-III completes |
| 5 | Utilities | Drive sync, push to GitHub | As needed |

**GitHub repo:** [sdimri7/Continual-RL](https://github.com/sdimri7/Continual-RL)  
**Requirements:** Colab GPU runtime (T4 or better), Google Drive for persistence

---
## Section 0: Environment Setup
Run all cells in this section every time the Colab runtime restarts.

In [ ]:
# Cell 0.1 -- GPU Check & Colab Detection
import subprocess, sys, os

try:
    import google.colab
    IN_COLAB = True
    print('Running on Google Colab')
except ImportError:
    IN_COLAB = False
    print('Not running on Colab -- some cells may need adjustment')

gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                           '--format=csv,noheader'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(f'GPU: {gpu_info.stdout.strip()}')
else:
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type > GPU.')

print(f'Python: {sys.version}')
print(f'CUDA available: {subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout.split("release ")[-1].split(",")[0] if subprocess.run(["which", "nvcc"], capture_output=True).returncode == 0 else "nvcc not found"}')

In [ ]:
# Cell 0.2 -- Mount Google Drive & Create Directory Structure
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Continual-RL'

drive_subdirs = ['runs', 'demos', 'llm_generated', 'checkpoints', 'eval_videos']
for d in drive_subdirs:
    os.makedirs(os.path.join(DRIVE_ROOT, d), exist_ok=True)
    print(f'  {DRIVE_ROOT}/{d}/')

print(f'\nDrive root: {DRIVE_ROOT}')

In [ ]:
# Cell 0.3 -- Clone Repo from GitHub
REPO_URL = 'https://github.com/sdimri7/Continual-RL.git'
PROJECT_DIR = '/content/Continual-RL'
BRANCH = 'main'  # change to a feature branch if needed

if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    print('Repo already cloned, pulling latest changes...')
    !cd {PROJECT_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f'Cloning {REPO_URL} ...')
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git log --oneline -5
print(f'\nWorking directory: {os.getcwd()}')

In [ ]:
# Cell 0.4 -- Install Vulkan (required by ManiSkill GPU simulator)
if IN_COLAB:
    print('Installing Vulkan drivers...')
    !apt-get update -qq && apt-get install -y -qq mesa-vulkan-drivers libvulkan1 vulkan-tools > /dev/null 2>&1
    !vulkaninfo --summary 2>/dev/null | head -20 || echo 'Vulkan info not available (software rendering will be used)'
else:
    print('Skipping Vulkan install -- not on Colab')

In [ ]:
# Cell 0.5 -- Install Python Dependencies
print('Installing dependencies (this takes 2-3 minutes on first run)...')
!pip install -q mani-skill
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers accelerate tyro tensorboard wandb opencv-python h5py openai einops

# Install the diffusion_policy package in editable mode
!cd {PROJECT_DIR}/official_diffusion_policy && pip install -q -e .

print('\nInstalled packages:')
!pip show mani-skill torch diffusers 2>/dev/null | grep -E '^(Name|Version)' | paste - -

In [ ]:
# Cell 0.6 -- Create Symlinks (Project <-> Drive) & Set API Keys
import shutil

# Symlink mapping: project_path -> drive_path
symlinks = {
    os.path.join(PROJECT_DIR, 'runs'): os.path.join(DRIVE_ROOT, 'runs'),
    os.path.join(PROJECT_DIR, 'llm_reward_gen', 'generated'): os.path.join(DRIVE_ROOT, 'llm_generated'),
}

for proj_path, drive_path in symlinks.items():
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(proj_path):
        print(f'  Symlink exists: {proj_path} -> {os.readlink(proj_path)}')
    elif os.path.isdir(proj_path):
        # Move existing contents to Drive, then symlink
        for item in os.listdir(proj_path):
            src = os.path.join(proj_path, item)
            dst = os.path.join(drive_path, item)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(proj_path)
        os.symlink(drive_path, proj_path)
        print(f'  Created symlink: {proj_path} -> {drive_path}')
    else:
        os.symlink(drive_path, proj_path)
        print(f'  Created symlink: {proj_path} -> {drive_path}')

# Symlink ManiSkill demos to Drive so they persist across sessions
maniskill_demos = os.path.expanduser('~/.maniskill/demos')
drive_demos = os.path.join(DRIVE_ROOT, 'demos')
os.makedirs(os.path.dirname(maniskill_demos), exist_ok=True)
if not os.path.exists(maniskill_demos):
    os.symlink(drive_demos, maniskill_demos)
    print(f'  Created symlink: {maniskill_demos} -> {drive_demos}')
elif os.path.islink(maniskill_demos):
    print(f'  Symlink exists: {maniskill_demos} -> {os.readlink(maniskill_demos)}')
else:
    print(f'  {maniskill_demos} exists (not a symlink) -- demos stored locally')

print('\n--- API Keys ---')

# OpenRouter API key (required for T-III LLM reward generation)
try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    print('OPENROUTER_API_KEY loaded from Colab Secrets')
except Exception:
    if 'OPENROUTER_API_KEY' not in os.environ:
        import getpass
        os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Enter OPENROUTER_API_KEY: ')
    else:
        print('OPENROUTER_API_KEY already set in environment')

# W&B API key (optional -- for experiment tracking)
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('WANDB_API_KEY loaded from Colab Secrets')
except Exception:
    print('WANDB_API_KEY not set (optional -- W&B tracking disabled)')

In [ ]:
# Cell 0.7 -- Smoke Test: Create PushT Environment & Step
import torch
import gymnasium as gym
import mani_skill.envs

print('Creating PushT-v1 with 4 envs (state obs, physx_cuda)...')
env = gym.make(
    'PushT-v1',
    num_envs=4,
    obs_mode='rgb',
    control_mode='pd_ee_delta_pose',
    sim_backend='physx_cuda',
)
obs, info = env.reset(seed=42)
print(f'Observation shape: {obs}')
print(f'Action space: {env.action_space}')

for _ in range(10):
    action = torch.from_numpy(env.action_space.sample())
    obs, rew, term, trunc, info = env.step(action)

print(f'After 10 steps -- reward: {rew.cpu().numpy()}, terminated: {term.cpu().numpy()}')
env.close()
print('Smoke test passed!')

---
## Section 1: T-I -- Imitation Learning (Diffusion Policy)

Train a state-based Diffusion Policy on expert Push-T demonstrations.  
**Time:** ~2-4 hours for 50k iterations on T4 GPU.  
**Output:** `runs/<exp_name>/best_eval_success_once.pt`

In [ ]:
# Cell 1.1 -- Download Push-T Demonstrations
import os

demo_check = os.path.expanduser('~/.maniskill/demos/PushT-v1/rl')
if os.path.exists(demo_check):
    print(f'Demos already downloaded: {demo_check}')
else:
    print('Downloading PushT-v1 demonstrations...')
    !python -m mani_skill.utils.download_demo "PushT-v1"
    print('Done.')

In [ ]:
# Cell 1.2 -- Preprocess Demos (replay trajectory to state obs format)
import os

DEMO_BASE = os.path.expanduser('~/.maniskill/demos/PushT-v1/rl')
RAW_TRAJ = os.path.join(DEMO_BASE, 'trajectory.none.pd_ee_delta_pose.physx_cuda.h5')
CONVERTED_TRAJ = os.path.join(DEMO_BASE, 'trajectory.rgb.pd_ee_delta_pose.physx_cuda.h5')

if os.path.exists(CONVERTED_TRAJ):
    print(f'Converted demos already exist: {CONVERTED_TRAJ}')
else:
    print('Converting demos to state + pd_joint_delta_pos format...')
    !python -m mani_skill.trajectory.replay_trajectory \
        --traj-path {RAW_TRAJ} \
        --use-env-states \
        -c pd_ee_delta_pose \
        -o rgb \
        --save-traj \
        --num-envs 10 \
        -b physx_cuda
    print('Done.')

print(f'Demo path for training: {CONVERTED_TRAJ}')

In [ ]:
# Cell 1.3 -- Train Diffusion Policy
#
# RESUME INSTRUCTIONS: If Colab disconnects during training, checkpoints are
# saved on Drive every 5000 iterations. To resume:
#   1. Re-run Section 0 (cells 0.1-0.7)
#   2. Set RESUME_FROM below to the latest checkpoint path
#   3. Re-run this cell

import os

EXP_NAME = 'diffusion_policy-PushT-v1-state'
DEMO_PATH = os.path.expanduser(
    '~/.maniskill/demos/PushT-v1/rl/trajectory.rgb.pd_ee_delta_pose.physx_cuda.h5'
)

!cd {PROJECT_DIR}/official_diffusion_policy && python train_rgbd.py \
    --env-id PushT-v1 \
    --demo-path {DEMO_PATH} \
    --control-mode pd_ee_delta_pose \
    --obs-mode rgb \
    --sim-backend physx_cuda \
    --num-demos 100 \
    --total_iters 50000 \
    --batch_size 1024 \
    --lr 1e-4 \
    --obs_horizon 2 \
    --act_horizon 1 \
    --pred_horizon 16 \
    --max_episode_steps 150 \
    --num_eval_envs 100 \
    --num_eval_episodes 100 \
    --eval_freq 5000 \
    --save_freq 5000 \
    --log_freq 500 \
    --capture-video \
    --exp-name {EXP_NAME}

In [ ]:
# Cell 1.4 -- TensorBoard (view training curves)
%load_ext tensorboard
%tensorboard --logdir {PROJECT_DIR}/runs/

In [ ]:
# Cell 1.5 -- Locate Best Checkpoint
import glob

EXP_NAME = 'diffusion_policy-PushT-v1-state'
run_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'runs', f'{EXP_NAME}*')))

if not run_dirs:
    print('No training runs found. Run Cell 1.3 first.')
else:
    run_dir = run_dirs[-1]
    ckpts = glob.glob(os.path.join(run_dir, '*.pt'))
    best_ckpts = [c for c in ckpts if 'best' in os.path.basename(c)]
    if best_ckpts:
        BEST_CKPT = best_ckpts[0]
    elif ckpts:
        BEST_CKPT = sorted(ckpts)[-1]
    else:
        BEST_CKPT = None

    print(f'Run directory: {run_dir}')
    print(f'Best checkpoint: {BEST_CKPT}')
    print(f'\nAll checkpoints:')
    for c in sorted(ckpts):
        size_mb = os.path.getsize(c) / 1e6
        print(f'  {os.path.basename(c)} ({size_mb:.1f} MB)')

---
## Section 2: T-II -- Failure Analysis

Evaluate the trained diffusion policy, identify systematic failure modes,  
and select representative failure videos for T-III reward generation.

**Failure mode categories:**
- **rotation_failure**: T-block near goal position but wrong orientation
- **overshoot**: T-block pushed past the goal
- **stuck**: Robot barely moves the T-block

In [ ]:
# Cell 2.1 -- Run Evaluation Rollouts & Collect Metrics
#
# Loads the best diffusion policy checkpoint and runs 250 evaluation episodes,
# recording per-episode T-block final position, rotation, and success.

import sys, os, glob, torch, numpy as np, pandas as pd
import gymnasium as gym
import mani_skill.envs

sys.path.insert(0, os.path.join(PROJECT_DIR, 'official_diffusion_policy'))
from diffusion_policy.make_env import make_eval_envs
from diffusion_policy.evaluate import evaluate
from diffusion_policy.conditional_unet1d import ConditionalUnet1D
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from dataclasses import dataclass, field
from typing import Optional, List

# --- Configuration ---
EXP_NAME = 'diffusion_policy-PushT-v1-state'
NUM_EVAL_ENVS = 50
NUM_EVAL_EPISODES = 250

# Find checkpoint
run_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'runs', f'{EXP_NAME}*')))
if not run_dirs:
    raise FileNotFoundError('No training run found. Complete Section 1 first.')
run_dir = run_dirs[-1]
ckpts = glob.glob(os.path.join(run_dir, '*.pt'))
best_ckpts = [c for c in ckpts if 'best' in os.path.basename(c)]
CKPT_PATH = best_ckpts[0] if best_ckpts else sorted(ckpts)[-1]
print(f'Loading checkpoint: {CKPT_PATH}')

# Load checkpoint
device = torch.device('cuda')
ckpt = torch.load(CKPT_PATH, map_location=device)

# Reconstruct the Agent class from train.py
# The checkpoint contains 'model' state_dict and 'args'
saved_args = ckpt.get('args', {})
obs_dim = ckpt.get('obs_dim', None)
act_dim = ckpt.get('act_dim', None)

# If obs/act dims not saved, infer from env
if obs_dim is None or act_dim is None:
    tmp_env = gym.make('PushT-v1', num_envs=1, obs_mode='state',
                       control_mode='pd_joint_delta_pos', sim_backend='physx_cuda')
    obs_dim = tmp_env.observation_space.shape[-1]
    act_dim = tmp_env.action_space.shape[-1]
    tmp_env.close()

print(f'obs_dim={obs_dim}, act_dim={act_dim}')
print(f'Evaluation: {NUM_EVAL_EPISODES} episodes on {NUM_EVAL_ENVS} parallel envs')
print('\nNote: Detailed failure analysis requires loading the full Agent class.')
print('For now, run the evaluation via the train script with --capture-video,')
print('then analyze the recorded videos and trajectory data in the next cells.')

In [ ]:
# Cell 2.2 -- Classify Failure Modes from Evaluation Videos/Logs
#
# If evaluation videos were captured in Cell 1.3 (--capture-video), analyze them here.
# Otherwise, manually identify failure modes from TensorBoard metrics.

import glob, os
import numpy as np

EXP_NAME = 'diffusion_policy-PushT-v1-state'
run_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'runs', f'{EXP_NAME}*')))
run_dir = run_dirs[-1] if run_dirs else None

if run_dir:
    video_dir = os.path.join(run_dir, 'videos')
    eval_video_dir = os.path.join(run_dir, 'eval_videos')

    videos = []
    for vdir in [video_dir, eval_video_dir]:
        if os.path.exists(vdir):
            videos.extend(glob.glob(os.path.join(vdir, '**/*.mp4'), recursive=True))

    print(f'Found {len(videos)} evaluation videos in {run_dir}')
    for v in sorted(videos)[:10]:
        print(f'  {os.path.relpath(v, PROJECT_DIR)}')
    if len(videos) > 10:
        print(f'  ... and {len(videos)-10} more')
else:
    print('No run directory found. Complete Section 1 first.')

print('\n--- Failure Mode Classification Guide ---')
print('Watch the evaluation videos and classify failures:')
print('  rotation_failure: T-block near goal XY but wrong angle (>45 deg off)')
print('  overshoot:        T-block pushed past the goal position')
print('  stuck:            Robot barely moves the T-block')
print('  partial:          T-block partway to goal, episode times out')
print('\nSelect 1-2 representative failure videos per mode for T-III.')

In [ ]:
# Cell 2.3 -- Display Evaluation Videos
from base64 import b64encode
from IPython.display import HTML, display
import glob, os

def show_video(path, width=480):
    """Display an MP4 video inline in the notebook."""
    if not os.path.exists(path):
        print(f'Video not found: {path}')
        return
    mp4 = open(path, 'rb').read()
    b64 = b64encode(mp4).decode()
    display(HTML(f'''
        <div style="margin: 10px 0;">
            <p><strong>{os.path.basename(path)}</strong></p>
            <video width="{width}" controls>
                <source src="data:video/mp4;base64,{b64}" type="video/mp4">
            </video>
        </div>
    '''))

# Display first few evaluation videos
EXP_NAME = 'diffusion_policy-PushT-v1-state'
run_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'runs', f'{EXP_NAME}*')))

if run_dirs:
    run_dir = run_dirs[-1]
    videos = []
    for vdir in ['videos', 'eval_videos']:
        vpath = os.path.join(run_dir, vdir)
        if os.path.exists(vpath):
            videos.extend(sorted(glob.glob(os.path.join(vpath, '**/*.mp4'), recursive=True)))

    if videos:
        print(f'Showing first 4 of {len(videos)} videos:')
        for v in videos[:4]:
            show_video(v)
    else:
        print('No videos found. Re-run Cell 1.3 with --capture-video.')
else:
    print('No run directory found.')

In [ ]:
# Cell 2.4 -- Select Failure Videos for T-III
#
# After watching the videos above, set the paths to representative failure videos.
# These will be sent to the LLM in Section 3.

# EDIT THESE PATHS after watching the evaluation videos:
ROTATION_FAILURE_VIDEO = ''  # e.g., 'runs/.../eval_videos/0-0.mp4'
OVERSHOOT_FAILURE_VIDEO = '' # e.g., 'runs/.../eval_videos/0-3.mp4'

for name, path in [('Rotation failure', ROTATION_FAILURE_VIDEO),
                    ('Overshoot failure', OVERSHOOT_FAILURE_VIDEO)]:
    if path:
        exists = os.path.exists(os.path.join(PROJECT_DIR, path))
        print(f'{name}: {path} [{"OK" if exists else "NOT FOUND"}]')
    else:
        print(f'{name}: <not set -- edit this cell>')

---
## Section 3: T-III -- LLM Reward & Episode Config Generation

Send a failure video to an LLM (Claude via OpenRouter) and receive:
1. A **dense reward function** (`reward_*.py`) that provides gradient signal for the failure mode
2. An **episode config** (`episode_config_*.py`) that biases episode initialization toward the failure regime

**Requires:** `OPENROUTER_API_KEY` set in Cell 0.6  
**Cost:** ~$0.05-0.20 per generation attempt (Claude Sonnet with vision)

In [ ]:
# Cell 3.1 -- Configure Failure Mode (edit these variables)

FAILURE_MODE = 'rotation_failure'

FAILURE_DESCRIPTION = (
    'The robot successfully pushes the T-block close to the goal XY position '
    '(within 0.03m) but fails to achieve the correct orientation. The T-block '
    'ends up rotated 60-120 degrees off the goal rotation, and the policy '
    'oscillates the end-effector around the block without correcting the orientation.'
)

QUANTITATIVE_CHARS = [
    'T-block final XY distance to goal < 0.04m but intersection < 90%',
    'Final rotation error |cos(z_euler_block - goal_z_rot) - 1| > 0.3',
    'Failure probability > 70% when initial T-block Z-rotation in [1.5, 3.5] rad',
]

BASELINE_SUCCESS_RATE = 62

# Path to a failure video (set in Cell 2.4 or update here)
VIDEO_PATH = ROTATION_FAILURE_VIDEO if ROTATION_FAILURE_VIDEO else ''

LLM_MODEL = 'anthropic/claude-sonnet-4'

print(f'Failure mode: {FAILURE_MODE}')
print(f'Video: {VIDEO_PATH or "<not set>"}')
print(f'Model: {LLM_MODEL}')
print(f'API key set: {"OPENROUTER_API_KEY" in os.environ}')

In [ ]:
# Cell 3.2 -- Generate Reward Function + Episode Config
#
# Option A: Use the pre-written config file (recommended for rotation_failure)
# Option B: Pass all arguments inline

USE_CONFIG_FILE = True  # Set to False to use inline arguments

if USE_CONFIG_FILE:
    config_path = os.path.join(PROJECT_DIR, 'llm_reward_gen', 'failure_configs',
                               f'{FAILURE_MODE}.json')
    if not os.path.exists(config_path):
        print(f'Config file not found: {config_path}')
        print('Set USE_CONFIG_FILE = False and use inline arguments instead.')
    else:
        cmd = f'python {PROJECT_DIR}/llm_reward_gen/run_generate.py'
        cmd += f' --config {config_path}'
        if VIDEO_PATH:
            cmd += f' --video-path {os.path.join(PROJECT_DIR, VIDEO_PATH)}'
        print(f'Running: {cmd}')
        !{cmd}
else:
    quant_args = ' '.join([f'"{q}"' for q in QUANTITATIVE_CHARS])
    !python {PROJECT_DIR}/llm_reward_gen/run_generate.py \
        --video-path {os.path.join(PROJECT_DIR, VIDEO_PATH)} \
        --failure-mode {FAILURE_MODE} \
        --failure-description "{FAILURE_DESCRIPTION}" \
        --quantitative-chars {quant_args} \
        --baseline-success-rate {BASELINE_SUCCESS_RATE} \
        --model {LLM_MODEL}

# List generated files
gen_dir = os.path.join(PROJECT_DIR, 'llm_reward_gen', 'generated')
print(f'\nGenerated files in {gen_dir}:')
if os.path.exists(gen_dir):
    for f in sorted(os.listdir(gen_dir)):
        if f.endswith('.py'):
            print(f'  {f}')

In [ ]:
# Cell 3.3 -- Validate Generated Code
import glob, os

gen_dir = os.path.join(PROJECT_DIR, 'llm_reward_gen', 'generated')

# Auto-detect latest generated files for this failure mode
reward_files = sorted(glob.glob(os.path.join(gen_dir, f'reward_{FAILURE_MODE}_v*.py')))
config_files = sorted(glob.glob(os.path.join(gen_dir, f'episode_config_{FAILURE_MODE}_v*.py')))

REWARD_CODE = reward_files[-1] if reward_files else ''
EPISODE_CONFIG = config_files[-1] if config_files else ''

print(f'Reward code:    {REWARD_CODE}')
print(f'Episode config: {EPISODE_CONFIG}')

if REWARD_CODE:
    cmd = f'python {PROJECT_DIR}/llm_reward_gen/run_validate.py --reward-code {REWARD_CODE}'
    if EPISODE_CONFIG:
        cmd += f' --episode-config {EPISODE_CONFIG}'
    print(f'\nRunning validation...')
    !{cmd}
else:
    print('\nNo generated files found. Run Cell 3.2 first.')

In [ ]:
# Cell 3.4 -- Display Generated Reward Function
if REWARD_CODE and os.path.exists(REWARD_CODE):
    print(f'=== {os.path.basename(REWARD_CODE)} ===')
    with open(REWARD_CODE) as f:
        print(f.read())
else:
    print('No reward code to display.')

if EPISODE_CONFIG and os.path.exists(EPISODE_CONFIG):
    print(f'\n=== {os.path.basename(EPISODE_CONFIG)} ===')
    with open(EPISODE_CONFIG) as f:
        print(f.read())
else:
    print('No episode config to display.')

In [ ]:
# Cell 3.5 -- (Optional) Generate for Second Failure Mode
#
# Uncomment and edit to generate reward for a second failure mode (e.g., overshoot).

# FAILURE_MODE_2 = 'overshoot'
# VIDEO_PATH_2 = OVERSHOOT_FAILURE_VIDEO  # set in Cell 2.4
#
# if VIDEO_PATH_2:
#     config_path = os.path.join(PROJECT_DIR, 'llm_reward_gen', 'failure_configs',
#                                f'{FAILURE_MODE_2}.json')
#     !python {PROJECT_DIR}/llm_reward_gen/run_generate.py \
#         --config {config_path} \
#         --video-path {os.path.join(PROJECT_DIR, VIDEO_PATH_2)}
#
#     # Validate
#     reward2 = sorted(glob.glob(os.path.join(gen_dir, f'reward_{FAILURE_MODE_2}_v*.py')))[-1]
#     config2 = sorted(glob.glob(os.path.join(gen_dir, f'episode_config_{FAILURE_MODE_2}_v*.py')))[-1]
#     !python {PROJECT_DIR}/llm_reward_gen/run_validate.py \
#         --reward-code {reward2} --episode-config {config2}
# else:
#     print('Set VIDEO_PATH_2 in Cell 2.4 first.')

print('Cell 3.5: Skipped (uncomment to generate for second failure mode)')

---
## Section 4: T-IV -- PPO Fine-tuning with LLM Reward

Fine-tune a fresh PPO policy using the LLM-generated dense reward and  
failure-biased episode initialization.

The custom environment `PushT-LLMReward-v1` loads the generated reward function  
and episode config at runtime. Training episodes are 70% failure-biased + 30% uniform.

**Time:** ~1-2 hours for 5M timesteps on T4 GPU  
**Output:** `runs/<exp_name>/final_ckpt.pt`  
**Evaluation:** Targeted (failure-biased) + Nominal (full distribution, forgetting check)

In [ ]:
# Cell 4.1 -- PPO Training with LLM Reward
#
# RESUME: If Colab disconnects, re-run Section 0, then re-run this cell.
# Checkpoints are saved every eval_freq (25) iterations to Drive.
#
# To resume from a checkpoint, add: --checkpoint runs/<name>/ckpt_<N>.pt

import glob, os

FAILURE_MODE = 'rotation_failure'
gen_dir = os.path.join(PROJECT_DIR, 'llm_reward_gen', 'generated')

# Auto-detect latest generated files
reward_files = sorted(glob.glob(os.path.join(gen_dir, f'reward_{FAILURE_MODE}_v*.py')))
config_files = sorted(glob.glob(os.path.join(gen_dir, f'episode_config_{FAILURE_MODE}_v*.py')))

if not reward_files:
    raise FileNotFoundError(f'No reward files for {FAILURE_MODE}. Complete Section 3 first.')

REWARD_CODE_PATH = reward_files[-1]
EPISODE_CONFIG_PATH = config_files[-1] if config_files else ''

print(f'Reward:  {REWARD_CODE_PATH}')
print(f'Config:  {EPISODE_CONFIG_PATH}')
print(f'Failure: {FAILURE_MODE}')
print(f'\nStarting PPO training (5M steps, 256 envs)...\n')

cmd = f'python {PROJECT_DIR}/ppo/ppo_llm_reward.py'
cmd += f' --reward-code-path {REWARD_CODE_PATH}'
if EPISODE_CONFIG_PATH:
    cmd += f' --episode-config-path {EPISODE_CONFIG_PATH}'
cmd += f' --failure-mode {FAILURE_MODE}'
cmd += ' --num-envs 256'           # reduced from 512 for T4 GPU
cmd += ' --total-timesteps 5000000'
cmd += ' --gamma 0.8'
cmd += ' --gae-lambda 0.9'
cmd += ' --num-steps 50'
cmd += ' --update-epochs 4'
cmd += ' --learning-rate 3e-4'
cmd += ' --eval-nominal'            # also evaluate on full distribution
cmd += ' --capture-video'
cmd += ' --save-model'
cmd += ' --seed 1'

!{cmd}

In [ ]:
# Cell 4.2 -- PPO Training Curves
%load_ext tensorboard
%tensorboard --logdir {PROJECT_DIR}/runs/

In [ ]:
# Cell 4.3 -- Display PPO Evaluation Videos
import glob, os
from base64 import b64encode
from IPython.display import HTML, display

def show_video(path, width=480):
    if not os.path.exists(path):
        print(f'Video not found: {path}')
        return
    mp4 = open(path, 'rb').read()
    b64 = b64encode(mp4).decode()
    display(HTML(f'''
        <div style="margin: 10px 0;">
            <p><strong>{os.path.basename(path)}</strong></p>
            <video width="{width}" controls>
                <source src="data:video/mp4;base64,{b64}" type="video/mp4">
            </video>
        </div>
    '''))

# Find PPO run directories
ppo_runs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'runs', f'*{FAILURE_MODE}*')))
if ppo_runs:
    ppo_run = ppo_runs[-1]
    videos = sorted(glob.glob(os.path.join(ppo_run, '**/*.mp4'), recursive=True))
    print(f'PPO run: {ppo_run}')
    print(f'Found {len(videos)} videos\n')
    for v in videos[:6]:
        show_video(v)
else:
    print('No PPO runs found. Run Cell 4.1 first.')

In [ ]:
# Cell 4.4 -- (Optional) PPO for Second Failure Mode
#
# Uncomment to run PPO on a second failure mode.

# FAILURE_MODE_2 = 'overshoot'
# reward2 = sorted(glob.glob(os.path.join(gen_dir, f'reward_{FAILURE_MODE_2}_v*.py')))[-1]
# config2 = sorted(glob.glob(os.path.join(gen_dir, f'episode_config_{FAILURE_MODE_2}_v*.py')))[-1]
#
# !python {PROJECT_DIR}/ppo/ppo_llm_reward.py \
#     --reward-code-path {reward2} \
#     --episode-config-path {config2} \
#     --failure-mode {FAILURE_MODE_2} \
#     --num-envs 256 \
#     --total-timesteps 5000000 \
#     --gamma 0.8 --gae-lambda 0.9 --num-steps 50 \
#     --eval-nominal --capture-video --save-model --seed 1

print('Cell 4.4: Skipped (uncomment to train on second failure mode)')

In [ ]:
# Cell 4.5 -- Final Results Comparison
import glob, os
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_final_metric(log_dir, tag):
    """Read the last value of a scalar from TensorBoard logs."""
    try:
        ea = EventAccumulator(log_dir)
        ea.Reload()
        if tag in ea.scalars.Keys():
            events = ea.scalars.Items(tag)
            return events[-1].value if events else None
    except Exception:
        pass
    return None

print('=== Final Results ===')
print(f'{"Run":<50} {"Success Rate":>15}')
print('-' * 67)

all_runs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'runs', '*')))
for run_dir in all_runs:
    if not os.path.isdir(run_dir):
        continue
    run_name = os.path.basename(run_dir)

    # Try common metric names
    for tag in ['eval/success_once', 'eval/success_rate',
                'eval/targeted/success_once', 'eval/nominal/success_once',
                'charts/eval_success_once']:
        val = get_final_metric(run_dir, tag)
        if val is not None:
            metric_name = tag.split('/')[-1]
            print(f'{run_name[:50]:<50} {val:>14.1%} ({metric_name})')

print('\n(If no metrics appear, check that training completed and TensorBoard logs exist)')

---
## Section 5: Utilities

In [ ]:
# Cell 5.1 -- Manual Sync to Google Drive
#
# Use this if symlinks weren't set up or you want to force a backup.

import shutil, os

sync_pairs = [
    (os.path.join(PROJECT_DIR, 'runs'), os.path.join(DRIVE_ROOT, 'runs')),
    (os.path.join(PROJECT_DIR, 'llm_reward_gen', 'generated'), os.path.join(DRIVE_ROOT, 'llm_generated')),
]

for src, dst in sync_pairs:
    if os.path.exists(src) and not os.path.islink(src):
        print(f'Syncing {src} -> {dst} ...')
        os.makedirs(dst, exist_ok=True)
        !rsync -av --progress {src}/ {dst}/
    elif os.path.islink(src):
        print(f'  {src} is a symlink to Drive -- no sync needed')
    else:
        print(f'  {src} does not exist -- nothing to sync')

print('\nSync complete.')

In [ ]:
# Cell 5.2 -- Push Results to GitHub
#
# To push from Colab, you need a GitHub Personal Access Token (PAT).
# 1. Create one at: https://github.com/settings/tokens
# 2. Store it in Colab Secrets as 'GITHUB_TOKEN'
# 3. Run this cell

import os

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')

if GITHUB_TOKEN:
    # Set remote URL with token for authentication
    !cd {PROJECT_DIR} && git remote set-url origin https://{GITHUB_TOKEN}@github.com/sdimri7/Continual-RL.git

    # Add generated artifacts (not large model files)
    !cd {PROJECT_DIR} && git add llm_reward_gen/generated/*.py llm_reward_gen/docs/ colab_run.ipynb
    !cd {PROJECT_DIR} && git status

    # Commit and push
    !cd {PROJECT_DIR} && git commit -m "Add Colab training results and generated reward functions" || echo "Nothing to commit"
    !cd {PROJECT_DIR} && git push origin {BRANCH}
    print('\nPushed to GitHub.')

    # Reset URL to not expose token in config
    !cd {PROJECT_DIR} && git remote set-url origin https://github.com/sdimri7/Continual-RL.git
else:
    print('No GITHUB_TOKEN found.')
    print('To push from Colab, create a PAT at https://github.com/settings/tokens')
    print('and store it in Colab Secrets as GITHUB_TOKEN.')

In [ ]:
# Cell 5.3 -- Check Drive Storage Usage
import os

def dir_size(path):
    total = 0
    if os.path.islink(path):
        path = os.readlink(path)
    if os.path.isdir(path):
        for dirpath, dirnames, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if not os.path.islink(fp):
                    total += os.path.getsize(fp)
    return total

print(f'=== Google Drive Usage ({DRIVE_ROOT}) ===')
total = 0
for subdir in ['runs', 'demos', 'llm_generated', 'checkpoints', 'eval_videos']:
    path = os.path.join(DRIVE_ROOT, subdir)
    size = dir_size(path)
    total += size
    print(f'  {subdir:<20} {size/1e6:>10.1f} MB')
print(f'  {"TOTAL":<20} {total/1e6:>10.1f} MB')